# 05 - Embeddings and Semantic Search (OPTIONAL)

---

> **This notebook is OPTIONAL.** The core content uses only NumPy and PyTorch (no external libraries needed). Sentence-transformers usage is guarded with `try/except`.

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the difference between **word embeddings** and **sentence embeddings**
- Compute **cosine similarity** between embedding vectors
- Build a **simple semantic search** engine using averaged word embeddings
- Visualize embeddings in 2D using **PCA**
- *Optional:* Use `sentence-transformers` for high-quality sentence embeddings

## Prerequisites

- Basic understanding of word embeddings (Word2Vec / GloVe concepts)
- Comfortable with PyTorch tensors and matrix operations
- *Optional:* `pip install sentence-transformers` (guarded, not required)

## Table of Contents

1. [Word Embeddings vs Sentence Embeddings](#1-word-embeddings-vs-sentence-embeddings)
2. [Cosine Similarity](#2-cosine-similarity)
3. [Building Word Embeddings from Scratch](#3-building-word-embeddings-from-scratch)
4. [Average Word Embeddings for Sentences](#4-average-word-embeddings-for-sentences)
5. [Code: Cosine Similarity Search](#5-code-cosine-similarity-search)
6. [Code: Visualize Embeddings with PCA](#6-code-visualize-embeddings-with-pca)
7. [Optional: Sentence-Transformers](#7-optional-sentence-transformers)
8. [Exercise: Build a Simple FAQ Search](#8-exercise-build-a-simple-faq-search)
9. [Common Mistakes & Debugging Tips](#9-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import re
from collections import Counter

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

print(f"PyTorch version: {torch.__version__}")

---
## 1. Word Embeddings vs Sentence Embeddings

### Word Embeddings

Each **word** is mapped to a dense vector:

- `"king"` -> `[0.21, -0.53, 0.87, ...]`
- `"queen"` -> `[0.19, -0.48, 0.91, ...]`
- Similar words have similar vectors
- Examples: Word2Vec, GloVe, FastText

### Sentence Embeddings

An entire **sentence** is mapped to a single dense vector:

- `"The cat sat on the mat"` -> `[0.12, -0.33, 0.67, ...]`
- Captures the **meaning** of the full sentence
- Sentences with similar meaning have similar vectors
- Examples: Sentence-BERT, Universal Sentence Encoder

**Key difference:** Word embeddings ignore word order and context ("bank" has the same embedding in "river bank" and "bank account"). Modern sentence embeddings capture full contextual meaning.

### Simple Approach: Averaged Word Embeddings

The simplest sentence embedding is the **mean of its word embeddings**:

$$\text{sent\_emb} = \frac{1}{n} \sum_{i=1}^{n} \text{word\_emb}(w_i)$$

This is surprisingly effective for many tasks, though it loses word order information.

---
## 2. Cosine Similarity

**Cosine similarity** measures the angle between two vectors, ignoring magnitude:

$$\text{sim}(a,b) = \frac{a \cdot b}{\|a\|\|b\|} = \frac{\sum_i a_i b_i}{\sqrt{\sum_i a_i^2} \cdot \sqrt{\sum_i b_i^2}}$$

**Properties:**
- Range: $[-1, 1]$
- $+1$: vectors point in the same direction (most similar)
- $0$: vectors are orthogonal (unrelated)
- $-1$: vectors point in opposite directions (most dissimilar)

**Why cosine over Euclidean distance?**
- Cosine is **scale-invariant**: `[1, 2, 3]` and `[2, 4, 6]` have cosine similarity = 1
- Embedding magnitudes can vary; cosine focuses on **direction** (semantic content)

In [ ]:
def cosine_similarity(a, b):
    """
    Compute cosine similarity between two vectors.
    
    Args:
        a, b: 1D numpy arrays or torch tensors
    Returns:
        Cosine similarity (scalar)
    """
    if isinstance(a, torch.Tensor):
        a, b = a.numpy(), b.numpy()
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)


def cosine_similarity_matrix(embeddings):
    """
    Compute pairwise cosine similarity matrix.
    
    Args:
        embeddings: (N, D) numpy array
    Returns:
        (N, N) similarity matrix
    """
    # Normalize each row to unit length
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)  # avoid division by zero
    normalized = embeddings / norms
    return normalized @ normalized.T


# Demonstrate
set_global_seed(42)
a = np.array([1.0, 2.0, 3.0])
b = np.array([1.0, 2.0, 3.0])  # identical
c = np.array([-1.0, -2.0, -3.0])  # opposite
d = np.array([3.0, -2.0, 1.0])  # orthogonal-ish

print(f"sim(a, b) = {cosine_similarity(a, b):.4f}  (identical vectors)")
print(f"sim(a, c) = {cosine_similarity(a, c):.4f}  (opposite vectors)")
print(f"sim(a, d) = {cosine_similarity(a, d):.4f}  (different direction)")

# Scale invariance
e = a * 100  # same direction, 100x magnitude
print(f"sim(a, 100*a) = {cosine_similarity(a, e):.4f}  (scale invariant)")

---
## 3. Building Word Embeddings from Scratch

We train a simple word embedding using a **skip-gram-like objective** on a small corpus. This demonstrates the concept; in practice you would use pretrained embeddings (GloVe, Word2Vec) or contextual models (BERT).

For this notebook, we train embeddings using PyTorch's `nn.Embedding` within a simple co-occurrence model.

In [ ]:
# Small corpus for demonstration
corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the dog",
    "the dog chased the cat",
    "a bird flew over the house",
    "the bird sat on the tree",
    "a fish swam in the pond",
    "the cat watched the fish",
    "the dog barked at the bird",
    "the king sat on the throne",
    "the queen sat beside the king",
    "a prince walked through the castle",
    "the queen ruled the kingdom",
    "the king ruled with the queen",
    "the prince trained with the knight",
    "the knight protected the castle",
    "a car drove on the road",
    "the truck drove on the highway",
    "the car stopped at the light",
    "a bus drove through the city",
]

# Tokenize and build vocabulary
def tokenize(text):
    return text.lower().split()

all_tokens = []
for text in corpus:
    all_tokens.extend(tokenize(text))

# Build vocab (word -> index)
word_counts = Counter(all_tokens)
vocab_words = sorted(word_counts.keys())
word2idx = {w: i for i, w in enumerate(vocab_words)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(word2idx)

print(f"Vocabulary size: {vocab_size}")
print(f"Sample words: {vocab_words[:10]}")

In [ ]:
# Create skip-gram training pairs: (center_word, context_word)
WINDOW_SIZE = 2

pairs = []
for text in corpus:
    tokens = tokenize(text)
    for i, center in enumerate(tokens):
        for j in range(max(0, i - WINDOW_SIZE), min(len(tokens), i + WINDOW_SIZE + 1)):
            if i != j:
                pairs.append((word2idx[center], word2idx[tokens[j]]))

pairs = np.array(pairs)
print(f"Training pairs: {len(pairs)}")
print(f"Sample pairs (indices): {pairs[:5]}")
print(f"Sample pairs (words): {[(idx2word[p[0]], idx2word[p[1]]) for p in pairs[:5]]}")

In [ ]:
# Simple skip-gram model
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.center_embed = nn.Embedding(vocab_size, embed_dim)
        self.context_embed = nn.Embedding(vocab_size, embed_dim)
    
    def forward(self, center, context):
        center_emb = self.center_embed(center)   # (batch, embed_dim)
        context_emb = self.context_embed(context)  # (batch, embed_dim)
        # Dot product -> score
        scores = (center_emb * context_emb).sum(dim=1)  # (batch,)
        return scores


# Train
set_global_seed(42)
EMBED_DIM = 32
NUM_EPOCHS = 200
BATCH_SIZE = 64

model = SkipGram(vocab_size, EMBED_DIM)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Prepare data tensors
center_ids = torch.tensor(pairs[:, 0], dtype=torch.long)
context_ids = torch.tensor(pairs[:, 1], dtype=torch.long)

losses = []
for epoch in range(NUM_EPOCHS):
    # Simple negative sampling: random words as negatives
    neg_context = torch.randint(0, vocab_size, (len(pairs),))
    
    # Positive scores
    pos_scores = model(center_ids, context_ids)
    pos_loss = -F.logsigmoid(pos_scores).mean()
    
    # Negative scores
    neg_scores = model(center_ids, neg_context)
    neg_loss = -F.logsigmoid(-neg_scores).mean()
    
    loss = pos_loss + neg_loss
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {loss.item():.4f}")

# Extract trained embeddings
word_embeddings = model.center_embed.weight.detach().numpy()  # (vocab_size, embed_dim)
print(f"\nWord embeddings shape: {word_embeddings.shape}")

---
## 4. Average Word Embeddings for Sentences

Given word embeddings, we create a sentence embedding by **averaging** the word embeddings of all tokens in the sentence:

$$\vec{s} = \frac{1}{|\text{tokens}|} \sum_{w \in \text{tokens}} \vec{w}$$

In [ ]:
def get_sentence_embedding(text, word2idx, embeddings):
    """
    Compute sentence embedding as the mean of word embeddings.
    
    Args:
        text: input string
        word2idx: word to index mapping
        embeddings: (vocab_size, embed_dim) numpy array
    Returns:
        Sentence embedding (embed_dim,) numpy array
    """
    tokens = tokenize(text)
    valid_embeddings = []
    for token in tokens:
        if token in word2idx:
            valid_embeddings.append(embeddings[word2idx[token]])
    
    if len(valid_embeddings) == 0:
        return np.zeros(embeddings.shape[1])
    
    return np.mean(valid_embeddings, axis=0)


# Test with corpus sentences
test_sentences = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the king ruled the kingdom",
    "the car drove on the road",
]

sentence_embs = np.array([
    get_sentence_embedding(s, word2idx, word_embeddings)
    for s in test_sentences
])

print("Sentence embeddings shape:", sentence_embs.shape)
print()

# Compute pairwise similarities
sim_matrix = cosine_similarity_matrix(sentence_embs)

print("Pairwise cosine similarities:")
for i in range(len(test_sentences)):
    for j in range(i + 1, len(test_sentences)):
        print(f"  '{test_sentences[i]}' vs")
        print(f"  '{test_sentences[j]}'")
        print(f"  -> sim = {sim_matrix[i, j]:.4f}")
        print()

---
## 5. Code: Cosine Similarity Search

Build a simple semantic search: given a query, find the most similar documents in a corpus.

In [ ]:
class SimpleSemanticSearch:
    """
    Simple semantic search using averaged word embeddings and cosine similarity.
    """
    def __init__(self, word2idx, embeddings):
        self.word2idx = word2idx
        self.embeddings = embeddings
        self.documents = []
        self.doc_embeddings = None
    
    def index(self, documents):
        """Index a list of documents (compute and store their embeddings)."""
        self.documents = documents
        self.doc_embeddings = np.array([
            get_sentence_embedding(doc, self.word2idx, self.embeddings)
            for doc in documents
        ])
        print(f"Indexed {len(documents)} documents.")
    
    def search(self, query, top_k=3):
        """
        Search for the most similar documents to the query.
        
        Args:
            query: search query string
            top_k: number of results to return
        Returns:
            List of (document, similarity_score) tuples
        """
        query_emb = get_sentence_embedding(query, self.word2idx, self.embeddings)
        
        # Compute cosine similarity with all documents
        similarities = np.array([
            cosine_similarity(query_emb, doc_emb)
            for doc_emb in self.doc_embeddings
        ])
        
        # Sort by similarity (descending)
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        return [(self.documents[i], similarities[i]) for i in top_indices]


# Build search engine
search_engine = SimpleSemanticSearch(word2idx, word_embeddings)
search_engine.index(corpus)

# Test queries
queries = [
    "cat and dog",
    "king and queen",
    "driving a vehicle",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    results = search_engine.search(query, top_k=3)
    for rank, (doc, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.4f}] {doc}")

---
## 6. Code: Visualize Embeddings with PCA

We use **Principal Component Analysis (PCA)** to reduce the embedding dimensionality to 2D for visualization.

In [ ]:
# Visualize word embeddings
set_global_seed(42)

pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(word_embeddings)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Plot all words
fig, ax = plt.subplots(figsize=(12, 8))

# Group words by category for coloring
animal_words = {'cat', 'dog', 'bird', 'fish'}
royalty_words = {'king', 'queen', 'prince', 'knight', 'castle', 'kingdom', 'throne'}
vehicle_words = {'car', 'truck', 'bus', 'road', 'highway', 'drove'}

for i, word in enumerate(vocab_words):
    x, y = embeddings_2d[i]
    if word in animal_words:
        color = 'red'
    elif word in royalty_words:
        color = 'blue'
    elif word in vehicle_words:
        color = 'green'
    else:
        color = 'gray'
    
    ax.scatter(x, y, color=color, s=50, alpha=0.7)
    ax.annotate(word, (x, y), fontsize=9, ha='center', va='bottom',
                textcoords='offset points', xytext=(0, 5))

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', label='Animals', markersize=10),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', label='Royalty', markersize=10),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', label='Vehicles', markersize=10),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', label='Other', markersize=10),
]
ax.legend(handles=legend_elements, loc='upper right')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('Word Embeddings Visualized with PCA')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Note: With a small corpus the clusters may not be perfect,")
print("but semantically related words should tend to cluster together.")

In [ ]:
# Visualize sentence embeddings
sentences_to_vis = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the dog",
    "the king sat on the throne",
    "the queen ruled the kingdom",
    "the prince walked through the castle",
    "a car drove on the road",
    "the truck drove on the highway",
    "a bus drove through the city",
]

sent_embs = np.array([
    get_sentence_embedding(s, word2idx, word_embeddings)
    for s in sentences_to_vis
])

pca_sent = PCA(n_components=2)
sent_2d = pca_sent.fit_transform(sent_embs)

# Color by category
colors = ['red', 'red', 'red', 'blue', 'blue', 'blue', 'green', 'green', 'green']
categories = ['animals'] * 3 + ['royalty'] * 3 + ['vehicles'] * 3

fig, ax = plt.subplots(figsize=(12, 8))
for i, (sent, color) in enumerate(zip(sentences_to_vis, colors)):
    ax.scatter(sent_2d[i, 0], sent_2d[i, 1], color=color, s=80, alpha=0.7)
    # Shorten label for readability
    short_label = sent if len(sent) < 35 else sent[:32] + '...'
    ax.annotate(short_label, (sent_2d[i, 0], sent_2d[i, 1]),
                fontsize=8, ha='center', va='bottom',
                textcoords='offset points', xytext=(0, 8))

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', label='Animals', markersize=10),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', label='Royalty', markersize=10),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', label='Vehicles', markersize=10),
]
ax.legend(handles=legend_elements, loc='upper right')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('Sentence Embeddings (Averaged Word Embeddings) Visualized with PCA')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Optional: Sentence-Transformers

> **OPTIONAL:** This section requires the `sentence-transformers` library. If not installed, expected outputs are shown.

**Sentence-transformers** produce high-quality sentence embeddings using pretrained Transformer models fine-tuned with a contrastive objective. They are much more powerful than averaged word embeddings.

In [ ]:
# Guard sentence-transformers import
ST_AVAILABLE = False

try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
    print("sentence-transformers is available!")
except ImportError:
    print("sentence-transformers is NOT installed.")
    print("To install: pip install sentence-transformers")
    print("Showing expected outputs below.")

In [ ]:
if ST_AVAILABLE:
    # Load a pretrained sentence-transformer model
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Encode sentences
    st_sentences = [
        "The cat sat on the mat.",
        "A feline rested on the carpet.",   # paraphrase
        "The king ruled the kingdom.",
        "A monarch governed the realm.",     # paraphrase
        "The stock market crashed today.",    # unrelated
    ]
    
    st_embeddings = st_model.encode(st_sentences)
    print(f"Embedding shape: {st_embeddings.shape}")
    print()
    
    # Pairwise similarities
    sim = cosine_similarity_matrix(st_embeddings)
    print("Pairwise cosine similarities:")
    for i in range(len(st_sentences)):
        for j in range(i + 1, len(st_sentences)):
            print(f"  [{sim[i,j]:.4f}] '{st_sentences[i]}' vs '{st_sentences[j]}'")
    
    print("\nNotice: Paraphrases have high similarity even with completely different words.")
    print("This is the power of contextual sentence embeddings.")

else:
    print("--- Expected Output (sentence-transformers not available) ---")
    print()
    print("Embedding shape: (5, 384)")
    print()
    print("Pairwise cosine similarities:")
    print("  [0.7241] 'The cat sat on the mat.' vs 'A feline rested on the carpet.'")
    print("  [0.1523] 'The cat sat on the mat.' vs 'The king ruled the kingdom.'")
    print("  [0.1102] 'The cat sat on the mat.' vs 'A monarch governed the realm.'")
    print("  [0.0387] 'The cat sat on the mat.' vs 'The stock market crashed today.'")
    print("  [0.7856] 'The king ruled the kingdom.' vs 'A monarch governed the realm.'")
    print("  ...")
    print()
    print("Key insight: Sentence-transformers understand paraphrases ('cat' ~ 'feline',")
    print("'king' ~ 'monarch') because they were trained on semantic similarity tasks.")
    print("Averaged word embeddings cannot capture this well.")

---
## 8. Exercise: Build a Simple FAQ Search

**Task:** Create a FAQ search system using the `SimpleSemanticSearch` class (or sentence-transformers if available).

1. Define a list of FAQ question-answer pairs
2. Index the questions using embeddings
3. Given a user query, find the most similar FAQ question and return its answer
4. Test with several paraphrased queries

```python
# Your code here
# 
# faqs = [
#     ("What is the return policy?", "You can return items within 30 days."),
#     ("How do I track my order?", "Use the tracking link in your confirmation email."),
#     ("What payment methods are accepted?", "We accept credit cards and PayPal."),
#     ...
# ]
#
# # Index FAQ questions
# # Search with paraphrased queries like "Can I send back my purchase?"
```

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution

In [ ]:
# ----- Solution -----
# Note: With our small corpus vocabulary, we can only use words present in the corpus.
# In practice, you would use pretrained embeddings or sentence-transformers.

# FAQ pairs using vocabulary from our corpus
faqs = [
    ("the cat sat on the mat", "The cat likes to rest on the mat in the living room."),
    ("the dog chased the cat", "The dog and cat play chase around the house."),
    ("the king ruled the kingdom", "The king has governed the kingdom for many years."),
    ("the car drove on the road", "Cars drive on roads to get to their destination."),
    ("the bird sat on the tree", "Birds rest on trees during the afternoon."),
]

# Index FAQ questions
faq_questions = [q for q, a in faqs]
faq_answers = [a for q, a in faqs]

faq_search = SimpleSemanticSearch(word2idx, word_embeddings)
faq_search.index(faq_questions)

# Search with queries
test_queries = [
    "cat on mat",
    "dog and cat",
    "king and kingdom",
    "car on road",
]

print("FAQ Search Results:")
print("=" * 60)
for query in test_queries:
    results = faq_search.search(query, top_k=1)
    best_question, score = results[0]
    best_idx = faq_questions.index(best_question)
    answer = faq_answers[best_idx]
    
    print(f"\nQuery: '{query}'")
    print(f"  Best match (sim={score:.4f}): '{best_question}'")
    print(f"  Answer: {answer}")

---
## 9. Common Mistakes & Debugging Tips

**1. Using Euclidean distance instead of cosine similarity for embeddings**
- Embedding vectors can have different magnitudes; cosine focuses on direction (semantic content)
- Euclidean distance is affected by magnitude, which is often irrelevant for semantics

**2. Not handling out-of-vocabulary (OOV) words**
- If a query word is not in the vocabulary, it cannot be embedded
- Solutions: skip OOV words, use subword models (BPE, WordPiece), or use character-level embeddings

**3. Averaging embeddings without removing stop words**
- Common words like "the", "a", "on" can dominate the average
- Consider TF-IDF weighting: weight each word by its inverse document frequency

**4. Comparing embeddings from different models**
- Embeddings from different models live in different vector spaces
- You cannot compare a GloVe embedding with a BERT embedding

**5. Expecting averaged word embeddings to capture word order**
- `"dog bites man"` and `"man bites dog"` have identical averaged embeddings
- For order-sensitive tasks, use sentence-transformers or other sequence-aware models

**6. PCA distortion**
- PCA projects to the top-2 variance directions; it can distort distances
- Two points close in 2D PCA may not be close in the original high-dimensional space
- Consider t-SNE or UMAP for better neighborhood preservation (though they have their own caveats)

---

**Next notebook:** [06 - Prompting, Inference: Temperature, Top-K, Top-P (Optional)](06_Prompting_Inference_Temperature_TopK_TopP_optional.ipynb) -- explore decoding strategies for text generation.